# Cypress baseline for Electronics AI (plan F1)

1. **Runtime > Change runtime type > A100 GPU**, then save.
2. **Runtime > Run all**. Cell 2 asks for `cypress_in.zip` (from `pixi run cypress export`).
3. When the last cell finishes, `cypress_out.zip` downloads. Send it back.

The build pins Cypress's own stack: Python 3.10, PyTorch 1.13.1 + CUDA 11.7, nvcc 11.7 with gcc 11,
because Colab's default PyTorch needs C++17 while Cypress compiles as C++14. Every step stops
with a clear message if the GPU, CUDA or the CUDA kernels are missing; nothing falls back to the CPU.
First run takes about 25 minutes (CUDA toolkit download and the build).

In [ ]:
# 1. GPU check
import subprocess

r = subprocess.run(["nvidia-smi", "--query-gpu=name,driver_version,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True, check=False)
print(r.stdout or r.stderr)
assert r.returncode == 0 and r.stdout.strip(), "No GPU. Runtime > Change runtime type > A100 GPU"

In [ ]:
# 2. Input boards
import os
import zipfile

from google.colab import files

if not os.path.exists("/content/cypress_in.zip"):
    uploaded = files.upload()  # pick cypress_in.zip
    name = next(iter(uploaded))
    if name != "cypress_in.zip":
        os.replace(name, "/content/cypress_in.zip")
os.makedirs("/content/in", exist_ok=True)
zipfile.ZipFile("/content/cypress_in.zip").extractall("/content/in")
print(sorted(os.listdir("/content/in")))

In [ ]:
%%bash
# 3. System packages, gcc 11 and the CUDA 11.7 toolkit (toolkit only; the A100 driver is Colab's)
set -euo pipefail
export DEBIAN_FRONTEND=noninteractive
apt-get -qq update
apt-get -qq install -y bison flex libboost-all-dev libcairo2-dev gcc-11 g++-11 > /dev/null
if [ ! -x /usr/local/cuda-11.7/bin/nvcc ]; then
  wget -q https://developer.download.nvidia.com/compute/cuda/11.7.1/local_installers/cuda_11.7.1_515.65.01_linux.run -O /content/cuda117.run
  sh /content/cuda117.run --silent --toolkit --toolkitpath=/usr/local/cuda-11.7 --override --no-man-page
  rm -f /content/cuda117.run
fi
/usr/local/cuda-11.7/bin/nvcc --version | tail -n 2
gcc-11 --version | head -n 1

In [ ]:
%%bash
# 4. Python 3.10 environment with PyTorch 1.13.1 + CUDA 11.7
set -euo pipefail
cd /content
if [ ! -x /content/bin/micromamba ]; then
  curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xj bin/micromamba
fi
export MAMBA_ROOT_PREFIX=/content/mamba
[ -x /content/env/bin/python ] || /content/bin/micromamba create -y -q -p /content/env -c conda-forge python=3.10 pip
/content/env/bin/pip install -q torch==1.13.1+cu117 --extra-index-url https://download.pytorch.org/whl/cu117
/content/env/bin/pip install -q "numpy<2" scipy shapely matplotlib cairocffi pyunpack patool pkgconfig \
    "cmake>=3.21,<3.28" pyyaml
/content/env/bin/python -c "import torch; assert torch.cuda.is_available(), 'PyTorch sees no CUDA'; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"

In [ ]:
%%bash
# 5. Clone and build Cypress with its CUDA kernels (about 15 minutes)
set -euo pipefail
export PATH=/content/env/bin:/usr/local/cuda-11.7/bin:$PATH
export CUDA_HOME=/usr/local/cuda-11.7 CC=gcc-11 CXX=g++-11 CUDAHOSTCXX=g++-11
cd /content
[ -d Cypress/.git ] || git clone -q --recursive https://github.com/NVlabs/Cypress.git
cd Cypress
git submodule update --init --recursive -q
echo "Cypress $(git rev-parse HEAD)"
mkdir -p build && cd build
if ! cmake .. -DCMAKE_INSTALL_PREFIX=/content/Cypress/install -DCMAKE_BUILD_TYPE=Release \
      -DPython_EXECUTABLE=/content/env/bin/python -DPYTHON_EXECUTABLE=/content/env/bin/python \
      -DCUDA_TOOLKIT_ROOT_DIR=/usr/local/cuda-11.7 -DCUDA_HOST_COMPILER=/usr/bin/g++-11 \
      -DCUDA_ARCH_LIST="8.0" > cmake.log 2>&1; then
  tail -n 60 cmake.log; exit 1
fi
grep -i -E "cuda|torch" cmake.log | head -n 20 || true
if ! make -j"$(nproc)" > make.log 2>&1; then
  grep -n -i -m 40 "error" make.log; exit 1
fi
make install > install.log 2>&1
n_cuda=$(find /content/Cypress/install -name "*cuda*.so" | wc -l)
echo "CUDA kernel libraries: $n_cuda"
[ "$n_cuda" -gt 0 ] || { echo "No CUDA kernels were built: Cypress would run on the CPU. Send cmake.log."; exit 1; }

In [ ]:
%%writefile /content/cypress_run.py
# SPDX-License-Identifier: MIT
"""Place every board of /content/in with Cypress on the GPU; write /content/out/*.placed.json.

Input: ``<name>.json`` in the dictionary form of Cypress's ``pcb.PCB`` plus a
``fixed`` list (written by ``pixi run cypress export``).  The board goes through
Cypress's own BookshelfConverter with ``anonymous=False`` (the default would
replace reference designators), ``dreamplace/Placer.py`` places it on the GPU,
and the placed design is written back as a ``PCB.to_dict()``.
"""

import glob
import json
import os
import shutil
import subprocess
import sys
import time
import traceback

CYPRESS = os.environ.get("CYPRESS_DIR", "/content/Cypress")
INSTALL = os.path.join(CYPRESS, "install")
IN_DIR = os.environ.get("CYPRESS_IN", "/content/in")
OUT_DIR = os.environ.get("CYPRESS_OUT", "/content/out")
TIMEOUT_S = int(os.environ.get("CYPRESS_TIMEOUT", "3600"))
# defaults of test/tune/pcb-configspace.json in the Cypress repository
PCB_DEFAULTS = {"num_bins_x": 512, "num_bins_y": 512, "density_weight": 0.008, "gamma": 0.1318231577,
                "target_density": 1.0, "stop_overflow": 0.07}

sys.path.insert(0, os.path.join(CYPRESS, "pcb-util"))
from bookshelf_converter import BookshelfConverter
from pcb import PCB


def load_pcb(data):
    pcb = PCB()
    result = pcb.from_dict({k: v for k, v in data.items() if k != "fixed"})
    pcb = result if isinstance(result, PCB) else pcb
    fixed = set(data.get("fixed", []))
    for name, comp in pcb.components.items():
        comp.fixed = name in fixed
    return pcb


def run_one(path):
    name = os.path.basename(path)[:-5]
    with open(path, encoding="utf-8") as fh:
        data = json.load(fh)
    work = os.path.join(OUT_DIR, "work", name)
    bs = os.path.join(work, "bookshelf")
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(bs)
    row = {"name": name, "components": len(data["components"]), "fixed": len(data.get("fixed", []))}
    t0 = time.time()
    pcb = load_pcb(data)
    BookshelfConverter(design_name=name, target_folder=bs, filter_fixed_on_layer=None, movable_pages=None,
                       anonymous=False).to_bookshelf(pcb)
    pl_in = os.path.join(bs, f"{name}.pl")
    with open(pl_in, encoding="utf-8") as fh:
        row["fixed_in_bookshelf"] = sum(1 for line in fh if "/FIXED" in line)
    shutil.copyfile(pl_in, pl_in + ".orig")
    with open(os.path.join(CYPRESS, "test", "simple.json"), encoding="utf-8") as fh:
        cfg = json.load(fh)
    cfg.update(PCB_DEFAULTS)
    cfg.update({"aux_input": os.path.join(bs, f"{name}.aux"), "gpu": 1, "plot_flag": 0,
                "result_dir": os.path.join(work, "results"),
                "opt_num_bins": [dict(cfg["opt_num_bins"][0], x=PCB_DEFAULTS["num_bins_x"],
                                      y=PCB_DEFAULTS["num_bins_y"])]})
    cfg_path = os.path.join(work, "config.json")
    with open(cfg_path, "w", encoding="utf-8") as fh:
        json.dump(cfg, fh, indent=1)
    log_path = os.path.join(OUT_DIR, f"{name}.log")
    with open(log_path, "w", encoding="utf-8") as log:
        done = subprocess.run([sys.executable, os.path.join(INSTALL, "dreamplace", "Placer.py"), cfg_path],
                              cwd=INSTALL, stdout=log, stderr=subprocess.STDOUT, timeout=TIMEOUT_S, check=False)
    row["placer_exit"] = done.returncode
    with open(log_path, encoding="utf-8", errors="replace") as fh:
        text = fh.read()
    row["log_tail"] = text[-1500:]
    row["gpu_mentioned"] = ("gpu" in text.lower()) or ("cuda" in text.lower())
    pls = sorted(glob.glob(os.path.join(work, "results", "**", "*.pl"), recursive=True), key=os.path.getmtime)
    if done.returncode != 0 or not pls:
        raise RuntimeError(f"Placer.py exit {done.returncode}, {len(pls)} .pl files; see {name}.log")
    best = next((p for p in reversed(pls) if ".lg." in os.path.basename(p)), pls[-1])
    row["pl"] = os.path.relpath(best, work)
    shutil.copyfile(best, pl_in)
    placed = BookshelfConverter(design_name=name, target_folder=bs, filter_fixed_on_layer=None, movable_pages=None,
                                anonymous=False).from_bookshelf()
    with open(os.path.join(OUT_DIR, f"{name}.placed.json"), "w", encoding="utf-8") as fh:
        json.dump(placed.to_dict(), fh)
    shutil.copyfile(best, os.path.join(OUT_DIR, f"{name}.pl"))
    row["seconds"] = round(time.time() - t0, 1)
    row["status"] = "ok"
    return row


def main():
    import torch

    os.makedirs(OUT_DIR, exist_ok=True)
    summary = {"torch": torch.__version__, "torch_cuda": torch.version.cuda,
               "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               "cypress_commit": subprocess.run(["git", "-C", CYPRESS, "rev-parse", "HEAD"], capture_output=True, check=False,
                                                text=True).stdout.strip(),
               "boards": []}
    if not torch.cuda.is_available():
        raise SystemExit("CUDA is not available to PyTorch: refusing to run Cypress on the CPU")
    for path in sorted(glob.glob(os.path.join(IN_DIR, "*.json"))):
        if os.path.basename(path) == "manifest.json":
            continue
        try:
            row = run_one(path)
        except Exception as exc:  # noqa: BLE001
            row = {"name": os.path.basename(path), "status": "failed", "error": f"{type(exc).__name__}: {exc}",
                   "trace": traceback.format_exc()[-3000:]}
        summary["boards"].append(row)
        print(json.dumps({k: row.get(k) for k in ("name", "status", "seconds", "fixed", "fixed_in_bookshelf",
                                                  "gpu_mentioned", "error")}), flush=True)
        with open(os.path.join(OUT_DIR, "summary.json"), "w", encoding="utf-8") as fh:
            json.dump(summary, fh, indent=1)


if __name__ == "__main__":
    main()

In [ ]:
# 6. Place every board on the GPU
import subprocess

r = subprocess.run(["/content/env/bin/python", "/content/cypress_run.py"], capture_output=True, text=True, check=False)
print(r.stdout[-6000:])
print(r.stderr[-6000:])

In [ ]:
# 7. Package the results and download cypress_out.zip
import os
import zipfile

from google.colab import files

with zipfile.ZipFile("/content/cypress_out.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for name in sorted(os.listdir("/content/out")):
        path = os.path.join("/content/out", name)
        if os.path.isfile(path):
            zf.write(path, name)
    for log in ("cmake.log", "make.log"):
        p = os.path.join("/content/Cypress/build", log)
        if os.path.exists(p):
            zf.write(p, "build_" + log)
print(zipfile.ZipFile("/content/cypress_out.zip").namelist())
files.download("/content/cypress_out.zip")